# Chase Analytics Notebook


> Exploratory analysis of training logs. Load the Excel output, clean it, compute per-episode metrics, and visualize captures, actions, and trajectories.


## 0. Setup
Import libraries, set paths, and load helper utilities from `nb_utils.py`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path(__file__).parent if '__file__' in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RESULTS_PATH = PROJECT_ROOT / 'results' / 'chase_run.xlsx'

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from nb_utils import (
    RESULTS_PATH as DEFAULT_RESULTS_PATH,
    add_derived_columns,
    action_counts,
    capture_rate_by_run,
    combine_runs,
    episode_metrics,
    episode_trajectories,
    load_runs,
    plot_action_distribution,
    plot_capture_rate,
    plot_positions,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print(f'Notebook dir: {NOTEBOOK_DIR}')
print(f'Results file: {RESULTS_PATH}')

Notebook dir: c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook
Results file: c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\results\chase_run.xlsx


## 1. Load data
Read all sheets (runs) from the Excel log in `results/`.


In [ ]:
sheets = load_runs(RESULTS_PATH)
print('Sheets found:')
for name, df in sheets.items():
    print(f'  {name}: {len(df):,} rows')

## 2. Quick peek at a raw run
Inspect the first few rows of a sheet to see the raw columns. Adjust `run_name` if you want another sheet.


In [ ]:
run_name = next(iter(sheets))
sheets[run_name].head()

## 3. Combine runs and add derived columns
Merge all runs, add handy features (Manhattan distance, typed actions), and keep everything in one DataFrame.


In [ ]:
steps_df = add_derived_columns(combine_runs(sheets))
print(f'Combined rows: {len(steps_df):,}')
steps_df.head()


## 4. Episode-level metrics
Aggregate to per-episode stats (captured?, steps taken, total return) and summarize per run.


In [ ]:
ep_df = episode_metrics(steps_df)
summary_df = capture_rate_by_run(ep_df)
display(ep_df.head())
display(summary_df)


## 5. Capture rate by run
Simple bar plot of capture fraction per training run.


In [ ]:
plot_capture_rate(summary_df)
plt.show()


## 6. Action distributions
Stacked bars for how often each action is chosen by catcher and runner.


In [ ]:
catcher_actions = action_counts(steps_df, who='catcher')
runner_actions = action_counts(steps_df, who='runner')

plot_action_distribution(catcher_actions, title='Catcher action mix by run')
plt.show()

plot_action_distribution(runner_actions, title='Runner action mix by run')
plt.show()


## 7. Example trajectories
Pick a run and a couple of episodes to visualize positions over time (catcher vs runner).


In [ ]:
if summary_df.empty:
    print('No data available to plot trajectories.')
else:
    example_run = summary_df['run'].iloc[0]
    example_eps = ep_df.loc[ep_df['run'] == example_run, 'episode'].head(2).tolist()
    size_guess = int(steps_df['env_size'].dropna().iloc[0]) if 'env_size' in steps_df else 5
    traj_df = episode_trajectories(steps_df, example_run, example_eps)
    print(f'Plotting run={example_run}, episodes={example_eps}')
    plot_positions(traj_df, size=size_guess)
    plt.show()


## 8. Save processed data (optional)
Persist cleaned step-level or episode-level tables if you want to reuse them elsewhere.


In [ ]:
# Uncomment to save if needed.
# steps_df.to_parquet(NOTEBOOK_DIR / 'steps.parquet', index=False)
# ep_df.to_parquet(NOTEBOOK_DIR / 'episodes.parquet', index=False)
print('Processed data ready in variables: steps_df, ep_df, summary_df.')
